# Figure S2

Local $G_\mu$ ruggedness inference on $NK$ landscapes under uniform and biased mutation spectra. Each kernel has 625 local estimates per $(N,K)$ pair.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import subprocess
import sys
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import jax.random as jr
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec

from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = True
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
REPO_ROOT = Path.cwd()
print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")
print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save a figure in PDF, PNG, and EPS formats.

    Parameters:
    - fig: Figure
        Matplotlib figure to save.
    - stem: str
        Filename stem without an extension.
    - bbox_inches: str
        Bounding-box mode passed to Matplotlib.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: Axes, letter: str) -> None:
    """Add a bold manuscript panel letter.

    Parameters:
    - ax: Axes
        Axis receiving the annotation.
    - letter: str
        Panel letter.

    Returns:
    - None
        The annotation is added directly to ``ax``.
    """
    ax.text(-0.14, 1.10, letter, transform=ax.transAxes, fontsize=PANEL_LETTER_SIZE,
            fontweight="bold", va="top", ha="left")

from slide.data_generation import (
    nk_grid_pairs,
    ordered_unique_pairs,
    random_start,
    run_nk_start_averaged_diffusion,
)
from slide.direvo_functions import get_single_decay_rate
from slide_config import get_slide_data_dir

SLIDE_DATA_DIR = Path(get_slide_data_dir())
PROCESSED_PATH = PROCESSED_DATA_DIR / "figureS2_local_gmu_processed.pkl"
MUTATION_MATRIX_FILES = {
    "E. coli": REPO_ROOT / "other_data" / "normed_e_coli_matrix.npy",
    "A. thaliana": REPO_ROOT / "other_data" / "normed_a_thaliana_matrix.npy",
    "Human": REPO_ROOT / "other_data" / "normed_human_codon_matrix.npy",
}
N_VALUES: tuple[int, ...] = (10, 14, 18, 23, 27, 32, 36, 41, 45, 50)
NUM_ALLELES: int = 4
NUM_K_VALUES_PER_N: int = 10
NUM_LANDSCAPES_PER_PAIR: int = 25
NUM_STARTS_PER_LANDSCAPE: int = 25
NUM_POPULATION_REPLICATES: int = 5
POPULATION_SIZE: int = 2_500
NUM_GENERATIONS: int = 25
TOTAL_MUTATION_RATE: float = 0.5
RANDOM_SEED: int = 42
MODEL_LABELS: tuple[str, ...] = ("Uniform", "E. coli", "A. thaliana", "Human")
RAW_PATHS = {
    label: RAW_DATA_DIR / f"figureS2_local_gmu_{label.lower().replace(' ', '_').replace('.', '')}_raw.pkl"
    for label in MODEL_LABELS
}


## Figure S2 Raw Products

In [ ]:
uniform_kernel = (np.ones((NUM_ALLELES, NUM_ALLELES)) - np.eye(NUM_ALLELES)) / (NUM_ALLELES - 1)
mutation_kernels = {"Uniform": uniform_kernel}
mutation_kernels.update({
    label: np.asarray(np.load(path), dtype=float)
    for label, path in MUTATION_MATRIX_FILES.items()
})
for label, kernel in mutation_kernels.items():
    if kernel.shape != (NUM_ALLELES, NUM_ALLELES):
        raise ValueError(f"{label} kernel has shape {kernel.shape}.")
    if np.any(kernel < 0) or not np.allclose(kernel.sum(axis=1), 1.0):
        raise ValueError(f"{label} kernel is not row-stochastic.")
    adjacency = kernel > 0
    reachability = np.eye(NUM_ALLELES, dtype=bool)
    power = np.eye(NUM_ALLELES, dtype=bool)
    for _ in range(1, NUM_ALLELES):
        power = (power.astype(int) @ adjacency.astype(int)) > 0
        reachability |= power
    if not np.all(reachability):
        raise ValueError(f"{label} kernel is not irreducible.")
if not np.allclose(
    mutation_kernels["Uniform"],
    (np.ones((NUM_ALLELES, NUM_ALLELES)) - np.eye(NUM_ALLELES)) / 3,
):
    raise AssertionError("Uniform A=4 kernel is incorrect.")

raw_pairs = nk_grid_pairs((10, 50), NUM_K_VALUES_PER_N, K_start=0)
nk_pairs = ordered_unique_pairs(raw_pairs)
if len(nk_pairs) != 100 or any(k_value >= n_sites for n_sites, k_value in nk_pairs):
    raise AssertionError("Expected 100 valid Figure 3B-style NK pairs with K < N.")

raw_payloads: dict[str, dict[str, object]] = {}
if not PLOT_ONLY:
    pair_keys = jr.split(jr.PRNGKey(RANDOM_SEED), len(nk_pairs))
    for label in MODEL_LABELS:
        path = RAW_PATHS[label]
        if path.exists() and not OVERWRITE_RAW_PKL:
            raw_payloads[label] = load_pickle(path)
            continue
        kernel = mutation_kernels[label]
        trajectories = np.empty(
            (
                len(nk_pairs),
                NUM_LANDSCAPES_PER_PAIR,
                NUM_STARTS_PER_LANDSCAPE,
                NUM_POPULATION_REPLICATES,
                NUM_GENERATIONS,
            ),
            dtype=np.float32,
        )
        starts_padded = np.full(
            (
                len(nk_pairs),
                NUM_LANDSCAPES_PER_PAIR,
                NUM_STARTS_PER_LANDSCAPE,
                max(N_VALUES),
            ),
            -1,
            dtype=np.int8,
        )
        landscape_keys_saved = np.empty(
            (len(nk_pairs), NUM_LANDSCAPES_PER_PAIR, 2),
            dtype=np.uint32,
        )
        for pair_index, (pair_key, pair) in enumerate(zip(pair_keys, nk_pairs, strict=True)):
            n_sites, k_value = pair
            landscape_keys = jr.split(pair_key, NUM_LANDSCAPES_PER_PAIR)
            for landscape_index, landscape_key in enumerate(landscape_keys):
                start_keys = jr.split(
                    jr.fold_in(landscape_key, 100_000 + landscape_index),
                    NUM_STARTS_PER_LANDSCAPE,
                )
                starts = np.asarray([
                    random_start(
                        start_key,
                        n_sites=n_sites,
                        num_alleles=NUM_ALLELES,
                    )
                    for start_key in start_keys
                ])
                starts_padded[
                    pair_index, landscape_index, :, :n_sites
                ] = starts.astype(np.int8)
                landscape_keys_saved[
                    pair_index, landscape_index
                ] = np.asarray(landscape_key, dtype=np.uint32)
                trajectories[
                    pair_index, landscape_index
                ] = run_nk_start_averaged_diffusion(
                    rng_key=landscape_key,
                    trajectory_rng_key=landscape_key,
                    n_sites=n_sites,
                    k=k_value,
                    num_alleles=NUM_ALLELES,
                    starts=starts,
                    popsize=POPULATION_SIZE,
                    mutation_rate_per_site=TOTAL_MUTATION_RATE / n_sites,
                    num_reps_per_start=NUM_POPULATION_REPLICATES,
                    num_steps=NUM_GENERATIONS,
                    mutation_matrix=kernel,
                    return_replicates=True,
                )
        payload = {
            "data": {
                "fitness_trajectories": trajectories,
                "start_coordinates_padded": starts_padded,
                "landscape_keys": landscape_keys_saved,
                "mutation_kernel": kernel,
            },
            "params": {
                "model_label": label,
                "N_values": N_VALUES,
                "A": NUM_ALLELES,
                "nk_pairs": nk_pairs,
                "num_landscapes": NUM_LANDSCAPES_PER_PAIR,
                "num_starts": NUM_STARTS_PER_LANDSCAPE,
                "num_population_replicates": NUM_POPULATION_REPLICATES,
                "population_size": POPULATION_SIZE,
                "M": NUM_GENERATIONS,
                "total_mutation_rate": TOTAL_MUTATION_RATE,
                "mutation_rate_per_site": "total_mutation_rate / N",
                "seed": RANDOM_SEED,
                "start_policy": "random clonal genotype; no pre-optimisation",
            },
            "metadata": {
                "paper_reference": "Figure S2",
                "description": f"Replicate-level local G_mu support for {label} mutation.",
                "trajectory_axes": (
                    "NK_pair", "landscape", "start",
                    "population_replicate", "generation",
                ),
            },
        }
        save_pickle(payload, path)
        raw_payloads[label] = payload

    for label, payload in raw_payloads.items():
        values = np.asarray(payload["data"]["fitness_trajectories"])
        assert values.shape == (100, 25, 25, 5, 25)
        assert np.all(np.isfinite(values))
        assert tuple(tuple(value) for value in payload["params"]["nk_pairs"]) == tuple(nk_pairs)
    reference_starts = np.asarray(
        raw_payloads["Uniform"]["data"]["start_coordinates_padded"]
    )
    reference_keys = np.asarray(raw_payloads["Uniform"]["data"]["landscape_keys"])
    for label in MODEL_LABELS[1:]:
        assert np.array_equal(
            reference_starts,
            np.asarray(raw_payloads[label]["data"]["start_coordinates_padded"]),
        )
        assert np.array_equal(
            reference_keys,
            np.asarray(raw_payloads[label]["data"]["landscape_keys"]),
        )
    print("Validated replicate-level Figure S2 raw payloads.")
else:
    print("PLOT_ONLY=True: skipping Figure S2 raw generation/loading.")


## Figure S2 Processing

In [ ]:
def process_figure_s2_local_gmu(
    raw_by_label: dict[str, dict[str, object]],
) -> dict[str, object]:
    """Fit and bin local rho_2 estimates for all Figure S2 kernels.

    Parameters:
    - raw_by_label: dict[str, dict[str, object]]
        Replicate-level raw trajectory payloads keyed by mutation model.

    Returns:
    - dict[str, object]
        Start-level fits, binned summaries, parameters, and metadata.
    """
    output: dict[str, object] = {}
    reference_pairs: tuple[tuple[int, int], ...] | None = None
    for label in MODEL_LABELS:
        raw_payload = raw_by_label[label]
        trajectories = np.asarray(
            raw_payload["data"]["fitness_trajectories"],
            dtype=float,
        )
        pairs = tuple(
            (int(n_sites), int(k_value))
            for n_sites, k_value in raw_payload["params"]["nk_pairs"]
        )
        if reference_pairs is None:
            reference_pairs = pairs
        elif pairs != reference_pairs:
            raise ValueError("NK pairs differ across mutation models.")
        expected_shape = (
            len(pairs),
            NUM_LANDSCAPES_PER_PAIR,
            NUM_STARTS_PER_LANDSCAPE,
            NUM_POPULATION_REPLICATES,
            NUM_GENERATIONS,
        )
        if trajectories.shape != expected_shape:
            raise ValueError(
                f"{label} trajectories have shape {trajectories.shape}; "
                f"expected {expected_shape}."
            )

        f_mu = trajectories.mean(axis=3)
        g_mu = np.square(f_mu)
        fit_shape = g_mu.shape[:-1]
        rho_2_local = np.full(fit_shape, np.nan, dtype=float)
        fitted_constants = np.full(fit_shape, np.nan, dtype=float)
        fit_success = np.zeros(fit_shape, dtype=bool)
        fit_failures: list[dict[str, object]] = []
        for index in np.ndindex(fit_shape):
            curve = g_mu[index]
            try:
                if not np.all(np.isfinite(curve)) or np.isclose(curve[0], 0.0):
                    raise ValueError("G_mu is non-finite or begins at zero.")
                rate, constant = get_single_decay_rate(
                    curve,
                    mut=2.0 * TOTAL_MUTATION_RATE,
                    num_steps=NUM_GENERATIONS,
                )
                rho_2_local[index] = float(rate)
                fitted_constants[index] = float(constant)
                fit_success[index] = True
            except (RuntimeError, ValueError, FloatingPointError) as error:
                fit_failures.append({
                    "index": tuple(int(value) for value in index),
                    "error": str(error),
                })

        rho_nk = np.asarray(
            [(k_value + 1) / n_sites for n_sites, k_value in pairs],
            dtype=float,
        )
        rho_repeated = np.broadcast_to(
            rho_nk[:, None, None],
            rho_2_local.shape,
        )
        valid_rho = rho_repeated[fit_success]
        valid_estimates = rho_2_local[fit_success]
        bin_edges = np.linspace(0.0, 1.0, 11)
        bin_indices = np.clip(
            np.digitize(valid_rho, bin_edges, right=True) - 1,
            0,
            len(bin_edges) - 2,
        )
        binned_rho_nk = []
        binned_mean = []
        binned_std = []
        binned_counts = []
        for bin_index in range(len(bin_edges) - 1):
            in_bin = bin_indices == bin_index
            if not np.any(in_bin):
                continue
            binned_rho_nk.append(float(valid_rho[in_bin].mean()))
            binned_mean.append(float(valid_estimates[in_bin].mean()))
            binned_std.append(float(valid_estimates[in_bin].std()))
            binned_counts.append(int(in_bin.sum()))

        output[label] = {
            "f_mu": f_mu,
            "g_mu": g_mu,
            "rho_2_local": rho_2_local,
            "fitted_constants": fitted_constants,
            "fit_success": fit_success,
            "fit_failures": fit_failures,
            "rho_NK": rho_nk,
            "rho_2_local_pair_mean": np.nanmean(rho_2_local, axis=(1, 2)),
            "rho_2_local_pair_std": np.nanstd(rho_2_local, axis=(1, 2)),
            "binned_rho_NK": np.asarray(binned_rho_nk, dtype=float),
            "binned_mean": np.asarray(binned_mean, dtype=float),
            "binned_std": np.asarray(binned_std, dtype=float),
            "binned_counts": np.asarray(binned_counts, dtype=int),
            "bin_edges": bin_edges,
        }

    if reference_pairs is None:
        raise ValueError("No Figure S2 raw payloads were supplied.")
    return {
        "data": output,
        "params": {
            "N_values": N_VALUES,
            "A": NUM_ALLELES,
            "nk_pairs": reference_pairs,
            "num_landscapes": NUM_LANDSCAPES_PER_PAIR,
            "num_starts": NUM_STARTS_PER_LANDSCAPE,
            "num_population_replicates": NUM_POPULATION_REPLICATES,
            "local_estimates_per_pair": (
                NUM_LANDSCAPES_PER_PAIR * NUM_STARTS_PER_LANDSCAPE
            ),
            "population_size": POPULATION_SIZE,
            "M": NUM_GENERATIONS,
            "total_mutation_rate": TOTAL_MUTATION_RATE,
            "fit_mutation_scale": 2.0 * TOTAL_MUTATION_RATE,
            "seed": RANDOM_SEED,
            "models": MODEL_LABELS,
        },
        "metadata": {
            "paper_reference": "Figure S2",
            "description": (
                "Local G_mu decay-rate estimates from clonal, "
                "non-pre-optimised starts."
            ),
            "g_mu_definition": (
                "Population-replicate fitness curves are averaged per start, "
                "then squared."
            ),
            "standard_deviation_ddof": 0,
            "legacy_f_mu_cache_used": False,
        },
    }


if PROCESSED_PATH.exists() and (
    PLOT_ONLY or not OVERWRITE_PROCESSED_PKL
):
    figure_s2_payload = load_pickle(PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(
        f"PLOT_ONLY=True requires local G_mu payload {PROCESSED_PATH}. "
        "Set PLOT_ONLY=False to process the regenerated raw trajectories."
    )
else:
    missing_labels = [
        label for label in MODEL_LABELS if label not in raw_payloads
    ]
    if missing_labels:
        raise FileNotFoundError(
            f"Missing Figure S2 raw payloads for {missing_labels}."
        )
    figure_s2_payload = process_figure_s2_local_gmu(raw_payloads)
    save_pickle(figure_s2_payload, PROCESSED_PATH)

assert figure_s2_payload["metadata"]["legacy_f_mu_cache_used"] is False
assert figure_s2_payload["params"]["local_estimates_per_pair"] == 625
for label in MODEL_LABELS:
    panel = figure_s2_payload["data"][label]
    assert np.asarray(panel["rho_2_local"]).shape == (100, 25, 25)
    assert np.asarray(panel["fit_success"]).shape == (100, 25, 25)
print("Validated processed local G_mu Figure S2 payload.")


## Individual Panels A–D

In [ ]:
def plot_s2_panel(
    ax: Axes,
    payload: dict[str, object],
    model_label: str,
) -> None:
    """Plot one local G_mu ruggedness-accuracy panel.

    Parameters:
    - ax: Axes
        Axis receiving the plot.
    - payload: dict[str, object]
        Processed local G_mu Figure S2 payload.
    - model_label: str
        Display label identifying the mutation kernel.

    Returns:
    - None
        Panel artists are added directly to the axis.
    """
    panel = payload["data"][model_label]
    titles = {
        "Uniform": "Uniform mutation",
        "E. coli": r"$\mathit{E.\ coli}$",
        "A. thaliana": r"$\mathit{A.\ thaliana}$",
        "Human": "Human",
    }
    rate_label = (
        r"$\rho_2^{\mathrm{loc}}$"
        if model_label == "Uniform"
        else r"$\bar{\rho}_2^{\mathrm{loc}}$"
    )
    rho_nk = np.asarray(panel["binned_rho_NK"], dtype=float)
    means = np.asarray(panel["binned_mean"], dtype=float)
    standard_deviations = np.asarray(panel["binned_std"], dtype=float)
    ax.plot(
        rho_nk,
        means,
        "o-",
        linewidth=1.4,
        markersize=4,
        label=rate_label,
    )
    ax.fill_between(
        rho_nk,
        means - standard_deviations,
        means + standard_deviations,
        alpha=0.25,
        linewidth=0,
    )
    ax.plot(
        (0.0, 1.0),
        (0.0, 1.0),
        color="red",
        linestyle="--",
        alpha=0.55,
        label=r"$\rho_{NK}$",
    )
    ax.set_title(titles[model_label], fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel(r"$\rho_{NK}$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(rate_label, fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(0.0, 1.02)
    ax.set_ylim(0.0, 1.10)
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, alpha=0.18)


for letter, label in zip("ABCD", MODEL_LABELS, strict=True):
    fig, ax = plt.subplots(figsize=(3.2, 2.8), dpi=PANEL_DPI)
    plot_s2_panel(ax, figure_s2_payload, label)
    ax.legend(frameon=True, fontsize=7, loc="upper left")
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_S2{letter}")
    plt.show()


## Complete Figure S2

In [ ]:
fig, axes = plt.subplots(
    1,
    4,
    figsize=(12, 3.2),
    dpi=PANEL_DPI,
    constrained_layout=True,
    sharex=True,
    sharey=True,
)
for ax, letter, label in zip(axes, "ABCD", MODEL_LABELS, strict=True):
    plot_s2_panel(ax, figure_s2_payload, label)
    add_panel_letter(ax, letter)
    ax.legend(frameon=True, fontsize=7, loc="upper left")
if SAVE_FIGURES:
    save_figure(fig, "figure_S2")
plt.show()


## Manuscript Caption

```latex
\caption{\textbf{Local squared-fitness decay under uniform and biased mutation spectra on $NK$ landscapes.} Landscapes used a nucleotide alphabet ($A=4$), ten sequence lengths $N\in\{10,14,18,23,27,32,36,41,45,50\}$, and ten values of $K$ from $0$ to $N-1$. Panel A uses an unbiased mutation kernel with equal transition probabilities between the four nucleotide states; panels B--D use asymmetric, row-stochastic and irreducible $4\times4$ kernels labelled for \textit{E. coli}, \textit{A. thaliana}, and humans, respectively. For every $(N,K)$ pair and mutation kernel, 25 random clonal starting genotypes were sampled on each of 25 independently generated landscapes. Five mutation-only population replicates were simulated per start, using a population size of 2,500, 25 generations, and mutation rate $\theta=0.5$ per sequence per generation ($0.5/N$ per site). Replicate fitness trajectories were averaged per start to obtain $F_\mu$, after which $G_\mu=F_\mu^2$ was fitted with mutation scale $2\theta$ to obtain 625 local estimates per $(N,K)$ pair. Points and shaded regions show the mean and SD of local estimates within each $\rho_{NK}$ bin; red dashed lines denote $\rho_{NK}=(K+1)/N$.}
```
